In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import BallTree
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
training_df = pd.read_csv(
    filepath_or_buffer='training_faults_diagnostics.csv',
    low_memory=False
)
training_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1058069 entries, 0 to 1058068
Data columns (total 48 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   RecordID                   1058069 non-null  int64  
 1   EventTimeStamp             1058069 non-null  object 
 2   eventDescription           1002335 non-null  object 
 3   ecuSoftwareVersion         831493 non-null   object 
 4   ecuModel                   1002466 non-null  object 
 5   ecuMake                    1002466 non-null  object 
 6   ecuSource                  1058069 non-null  int64  
 7   spn                        1058069 non-null  int64  
 8   fmi                        1058069 non-null  int64  
 9   active                     1058069 non-null  bool   
 10  activeTransitionCount      1058069 non-null  int64  
 11  EquipmentID                1058069 non-null  object 
 12  MCTNumber                  1058069 non-null  int64  
 13  Latitude    

In [3]:
unnecessary_columns = [
    'RecordID',
    'EventTimeStamp',
    'LocationTimeStamp',
    'eventDescription',
    'ecuSoftwareVersion',
    'ecuModel',
    'ecuMake',
    'ecuSource',
    'EquipmentID',
    'MCTNumber',
    'NearServiceStation',
    'IsFullDerate',
    'Severity_Level',
    'Derate_Target_2.0-0.001',
    'Derate_Target_4.0-0.001',
    'Derate_Target_8.0-0.001'
]

In [4]:
# Drop columns that aren't needed for the model or aren't working witht the pipeline
training_df = training_df.drop(columns=unnecessary_columns)

## Identify features for imputing missing values

In [5]:
# Drop columns that have too many NaN values
nan_drop_threshold = 0.8

drop_nan_columns = training_df.columns[training_df.isna().mean() > nan_drop_threshold]
print(drop_nan_columns)

training_df = training_df.drop(columns=drop_nan_columns)

Index(['ServiceDistance', 'SwitchedBatteryVoltage'], dtype='object')


In [6]:
# Group categorical columns
categorical_columns = training_df.select_dtypes(include=["object", "bool"]).columns
print(categorical_columns)
print(len(categorical_columns))

# Group numeric columns
numeric_columns = training_df.select_dtypes(include=["int64", "float64"]).columns
print(numeric_columns)
print(len(numeric_columns))

Index(['active', 'CruiseControlActive', 'IgnStatus', 'ParkingBrake'], dtype='object')
4
Index(['spn', 'fmi', 'activeTransitionCount', 'Latitude', 'Longitude',
       'Severity_Level_Numeric', 'Derate_Target_12.0-0.001',
       'AcceleratorPedal', 'BarometricPressure', 'CruiseControlSetSpeed',
       'DistanceLtd', 'EngineCoolantTemperature', 'EngineLoad',
       'EngineOilPressure', 'EngineOilTemperature', 'EngineRpm',
       'EngineTimeLtd', 'FuelLevel', 'FuelLtd', 'FuelRate', 'FuelTemperature',
       'IntakeManifoldTemperature', 'LampStatus', 'Speed', 'Throttle',
       'TurboBoostPressure'],
      dtype='object')
26


In [7]:
# Group numeric columns by threshold
nan_low_threshold = 0.4

low_nan_numeric_columns = training_df[numeric_columns].columns[
    training_df[numeric_columns].isna().mean() <= nan_low_threshold
]
print(low_nan_numeric_columns)
print(len(low_nan_numeric_columns))

medium_nan_numeric_columns = training_df[numeric_columns].columns[
    training_df[numeric_columns].isna().mean() > nan_low_threshold
]
print(medium_nan_numeric_columns)
print(len(medium_nan_numeric_columns))

Index(['spn', 'fmi', 'activeTransitionCount', 'Latitude', 'Longitude',
       'Derate_Target_12.0-0.001', 'LampStatus'],
      dtype='object')
7
Index(['Severity_Level_Numeric', 'AcceleratorPedal', 'BarometricPressure',
       'CruiseControlSetSpeed', 'DistanceLtd', 'EngineCoolantTemperature',
       'EngineLoad', 'EngineOilPressure', 'EngineOilTemperature', 'EngineRpm',
       'EngineTimeLtd', 'FuelLevel', 'FuelLtd', 'FuelRate', 'FuelTemperature',
       'IntakeManifoldTemperature', 'Speed', 'Throttle', 'TurboBoostPressure'],
      dtype='object')
19


## Split training dataset

In [8]:
target = 'Derate_Target_12.0-0.001'

In [9]:
X = training_df.drop(columns=[target])
y = training_df[target]

In [10]:
y.isna().sum()

np.int64(0)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

## Create pipeline and fit model

In [12]:
categorical_pipe = Pipeline(
    steps=[
        ('categorical_imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder())
    ]
)

low_nan_numeric_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('low_nan_numeric_imputer', SimpleImputer(strategy='median'))
    ]
)

medium_nan_numeric_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('medium_nan_numeric_imputer', IterativeImputer(max_iter=20, random_state=30))
    ]
)

In [13]:
ct = ColumnTransformer(
    transformers=[
        ('categorical_pipe', categorical_pipe, categorical_columns),
        ('low_nan_numeric_pipe', low_nan_numeric_pipe, low_nan_numeric_columns.drop('Derate_Target_12.0-0.001')),
        ('medium_nan_numeric_pipe', medium_nan_numeric_pipe, medium_nan_numeric_columns)
    ]
)

In [14]:
pipe = Pipeline(
    steps=[
        ('transformer', ct),
        ('model', MLPClassifier(
            activation='relu',
            hidden_layer_sizes=(64,64,64)
        ))
    ]
).fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [15]:
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

In [16]:
print(classification_report(
    y_true=y_train,
    y_pred=y_pred_train
))

print(classification_report(
    y_true=y_test,
    y_pred=y_pred_test
))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    739352
           1       0.90      0.45      0.60      1296

    accuracy                           1.00    740648
   macro avg       0.95      0.72      0.80    740648
weighted avg       1.00      1.00      1.00    740648

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    316865
           1       0.62      0.24      0.35       556

    accuracy                           1.00    317421
   macro avg       0.81      0.62      0.67    317421
weighted avg       1.00      1.00      1.00    317421



In [17]:
training_cm = confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test
)
training_cm

array([[316785,     80],
       [   423,    133]])

In [18]:
print(f'Savings: {(training_cm[1][1]*4000) - (training_cm[0][1]*500)}')

Savings: 492000


In [23]:
testing_df = pd.read_csv(
    filepath_or_buffer='testing_faults_diagnostics.csv',
    low_memory=False
)

In [24]:
testing_target = testing_df[target]

In [25]:
testing_df = testing_df.drop(columns=unnecessary_columns)
testing_df = testing_df.drop(columns=drop_nan_columns)
testing_df = testing_df.drop(columns=target)
testing_df.columns

Index(['spn', 'fmi', 'active', 'activeTransitionCount', 'Latitude',
       'Longitude', 'Severity_Level_Numeric', 'AcceleratorPedal',
       'BarometricPressure', 'CruiseControlActive', 'CruiseControlSetSpeed',
       'DistanceLtd', 'EngineCoolantTemperature', 'EngineLoad',
       'EngineOilPressure', 'EngineOilTemperature', 'EngineRpm',
       'EngineTimeLtd', 'FuelLevel', 'FuelLtd', 'FuelRate', 'FuelTemperature',
       'IgnStatus', 'IntakeManifoldTemperature', 'LampStatus', 'ParkingBrake',
       'Speed', 'Throttle', 'TurboBoostPressure'],
      dtype='object')

In [26]:
testing_df['prediction'] = pipe.predict(testing_df)

In [27]:
print(classification_report(
    y_true=testing_target,
    y_pred=testing_df['prediction']
))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    128979
           1       0.04      0.01      0.02       287

    accuracy                           1.00    129266
   macro avg       0.52      0.50      0.51    129266
weighted avg       1.00      1.00      1.00    129266



In [29]:
testing_cm = confusion_matrix(
    y_true=testing_target,
    y_pred=testing_df['prediction']
)
testing_cm

array([[128911,     68],
       [   284,      3]])

In [30]:
print(f'Savings: {(testing_cm[1][1]*4000) - (testing_cm[0][1]*500)}')

Savings: -22000
